In [ ]:
!unzip -q /content/drive/MyDrive/HAM10000 -d /content/

In [ ]:
# Install libraries for google colab.
!pip install segmentation-models-pytorch torch torchmetrics albumentations

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# PyTorch libraries
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Segmentation Models PyTorch
import segmentation_models_pytorch as smp

# Albumentations for data augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Torchmetrics for evaluation metrics
from torchmetrics.classification import BinaryJaccardIndex, BinaryAccuracy

In [ ]:
# Set up the device to use GPU if avaliable, otherwise cpu.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, images_dir, masks_dir, image_names, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        # Get image and mask paths
        image_name = self.image_names[idx]
        image_path = os.path.join(self.images_dir, image_name)
        mask_name = image_name.replace('.jpg', '_segmentation.png')
        mask_path = os.path.join(self.masks_dir, mask_name)

        # Load image and mask
        image = np.array(Image.open(image_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'), dtype=np.float32)
        # Ensure mask is binary (0 or 1)
        mask[mask > 0] = 1.0

        # Apply transformations
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        else:
            image = TF.to_tensor(image)
            mask = torch.tensor(mask).unsqueeze(0)

        return image, mask

In [ ]:
# Define transformations for images and masks.
train_transforms = A.Compose([
    A.Resize(576, 448),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(576, 448),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# Paths to the images and masks directory.
images_dir = '/content/HAM10000/images/'
masks_dir = '/content/HAM10000/masks/'

# Get list of image names. Ignore files that are not .jpg or .png (as macOS adds hidden files).
image_names = [f for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]

# Split into training and validation sets
train_names, val_names = train_test_split(image_names, test_size=0.2, random_state=42)

# Create dataset instances
train_dataset = SkinLesionDataset(images_dir, masks_dir, train_names, transform=train_transforms)
val_dataset = SkinLesionDataset(images_dir, masks_dir, val_names, transform=val_transforms)

In [ ]:
# Create data loaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
# CBAM (Convolutional Block Attention Module)
# Code adapted from:
# https://github.com/xmu-xiaoma666/External-Attention-pytorch/blob/master/model/attention/CBAM.py
import torch
import torch.nn as nn
from torch.nn import init
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.se = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channel // reduction, channel, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_result = self.maxpool(x)
        avg_result = self.avgpool(x)
        max_out = self.se(max_result)
        avg_out = self.se(avg_result)
        output = self.sigmoid(max_out + avg_out)
        return output

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_result, _ = torch.max(x, dim=1, keepdim=True)
        avg_result = torch.mean(x, dim=1, keepdim=True)
        result = torch.cat([max_result, avg_result], dim=1)
        output = self.conv(result)
        output = self.sigmoid(output)
        return output

class CBAMBlock(nn.Module):
    def __init__(self, channel=512, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channel=channel, reduction=reduction)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                init.normal_(m.weight, std=0.001)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

    def forward(self, x):
        residual = x
        out = x * self.ca(x)
        out = out * self.sa(out)
        return out + residual

# Custom U-Net with ResNet34 Encoder & CBAM on Skip Connection.
import torchvision.models as models

class UNetResNet34_CBAM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        # Encoder: Pretrained ResNet34 backbone
        resnet = models.resnet34(pretrained=True)
        self.layer0 = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu
        )  # Output: 64 channels, stride 2
        self.layer1 = nn.Sequential(
            resnet.maxpool, resnet.layer1
        )  # Output: 64 channels, stride 4
        self.layer2 = resnet.layer2  # Output: 128 channels, stride 8
        self.layer3 = resnet.layer3  # Output: 256 channels, stride 16
        self.layer4 = resnet.layer4  # Output: 512 channels, stride 32 (bottleneck)

        # CBAM blocks for skip connections.
        # These are applied on the outputs of layer0, layer1, layer2, and layer3.
        self.cbam0 = CBAMBlock(channel=64, reduction=16, kernel_size=7)
        self.cbam1 = CBAMBlock(channel=64, reduction=16, kernel_size=7)
        self.cbam2 = CBAMBlock(channel=128, reduction=16, kernel_size=7)
        self.cbam3 = CBAMBlock(channel=256, reduction=16, kernel_size=7)
        # We do not add CBAM on the bottleneck (layer4)

        # Decoder Blocks: using transposed convolutions and conv layers.
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = nn.Sequential(
            nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = nn.Sequential(
            nn.Conv2d(128 + 128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.decoder0 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # ---------------------------
        # Encoder forward pass
        # ---------------------------
        x0 = self.layer0(x)    # [B, 64, H/2, W/2]
        x1 = self.layer1(x0)   # [B, 64, H/4, W/4]
        x2 = self.layer2(x1)   # [B, 128, H/8, W/8]
        x3 = self.layer3(x2)   # [B, 256, H/16, W/16]
        x4 = self.layer4(x3)   # [B, 512, H/32, W/32] -> Bottleneck

        # ---------------------------
        # Apply CBAM on skip connections
        # ---------------------------
        x0_att = self.cbam0(x0)
        x1_att = self.cbam1(x1)
        x2_att = self.cbam2(x2)
        x3_att = self.cbam3(x3)

        # ---------------------------
        # Decoder forward pass with CBAM-enhanced skip connections
        # ---------------------------
        d4 = self.upconv4(x4)                    # Upsample bottleneck [B, 256, H/16, W/16]
        d4 = torch.cat([d4, x3_att], dim=1)        # Concat with CBAM-ed features from layer3
        d4 = self.decoder3(d4)

        d3 = self.upconv3(d4)                     # Upsample [B, 128, H/8, W/8]
        d3 = torch.cat([d3, x2_att], dim=1)        # Concat with CBAM-ed features from layer2
        d3 = self.decoder2(d3)

        d2 = self.upconv2(d3)                     # Upsample [B, 64, H/4, W/4]
        d2 = torch.cat([d2, x1_att], dim=1)        # Concat with CBAM-ed features from layer1
        d2 = self.decoder1(d2)

        d1 = self.upconv1(d2)                     # Upsample [B, 64, H/2, W/2]
        d1 = torch.cat([d1, x0_att], dim=1)        # Concat with CBAM-ed features from layer0
        d1 = self.decoder0(d1)

        out = self.final_conv(d1)

        # Add final upsampling to match input size (576x448)
        out = nn.functional.interpolate(out, scale_factor=2, mode='bilinear', align_corners=True)
        return out

In [ ]:
model = UNetResNet34_CBAM(n_classes=1)
model = model.to(device)

In [ ]:
# Loss function
loss_fn = smp.losses.DiceLoss(mode='binary')

In [ ]:
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# Metrics
iou_metric = BinaryJaccardIndex().to(device)
accuracy_metric = BinaryAccuracy().to(device)

In [ ]:
num_epochs = 25
score_history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'val_iou': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    # Train phase
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        # Ensure that masks have shape [B, 1, H, W]
        masks = masks.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    # Validation phase
    model.eval()
    val_loss = 0.0
    total_iou = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device).float().unsqueeze(1)

            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item() * images.size(0)

            outputs = torch.sigmoid(outputs)
            preds = (outputs > 0.5).float()

            # Flatten tensors for metrics computation
            preds_flat = preds.view(-1)
            masks_flat = masks.view(-1)

            # Compute metrics
            iou = iou_metric(preds_flat, masks_flat)
            acc = accuracy_metric(preds_flat, masks_flat)

            total_iou += iou.item() * images.size(0)
            total_acc += acc.item() * images.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)
    avg_iou = total_iou / len(val_loader.dataset)
    avg_acc = total_acc / len(val_loader.dataset)

    # Save scores for each epoch
    score_history['epoch'].append(epoch + 1)
    score_history['train_loss'].append(avg_train_loss)
    score_history['val_loss'].append(avg_val_loss)
    score_history['val_iou'].append(avg_iou)
    score_history['val_accuracy'].append(avg_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}, Val IoU: {avg_iou:.4f}, Val Accuracy: {avg_acc:.4f}")

Epoch [1/25]
Train Loss: 0.1339
Val Loss: 0.0835, Val IoU: 0.8472, Val Accuracy: 0.9552
Epoch [2/25]
Train Loss: 0.0722
Val Loss: 0.0704, Val IoU: 0.8697, Val Accuracy: 0.9627
Epoch [3/25]
Train Loss: 0.0610
Val Loss: 0.0617, Val IoU: 0.8847, Val Accuracy: 0.9678
Epoch [4/25]
Train Loss: 0.0560
Val Loss: 0.0622, Val IoU: 0.8839, Val Accuracy: 0.9673
Epoch [5/25]
Train Loss: 0.0497
Val Loss: 0.0607, Val IoU: 0.8864, Val Accuracy: 0.9679
Epoch [6/25]
Train Loss: 0.0471
Val Loss: 0.0602, Val IoU: 0.8876, Val Accuracy: 0.9686
Epoch [7/25]
Train Loss: 0.0422
Val Loss: 0.0614, Val IoU: 0.8853, Val Accuracy: 0.9670
Epoch [8/25]
Train Loss: 0.0408
Val Loss: 0.0586, Val IoU: 0.8904, Val Accuracy: 0.9689
Epoch [9/25]
Train Loss: 0.0376
Val Loss: 0.0618, Val IoU: 0.8847, Val Accuracy: 0.9675
Epoch [10/25]
Train Loss: 0.0370
Val Loss: 0.0583, Val IoU: 0.8905, Val Accuracy: 0.9691
Epoch [11/25]
Train Loss: 0.0365
Val Loss: 0.0605, Val IoU: 0.8869, Val Accuracy: 0.9683
Epoch [12/25]
Train Loss: 0.03

In [ ]:
# Save the models weights
torch.save(model.state_dict(), "UNet_Resnet34_CBAM.pth")

In [ ]:
# Convert scores to a pandas dataframe and save it as a csv file
df_scores = pd.DataFrame(score_history)
df_scores.to_csv("CBAM_leasion_scores.csv", index=False)
print("Training scores saved.")

Training scores saved.
